## 1. Multi-Agent Orchestration

Coordinating multiple agents working toward common goals:
- **Agent Registry**: Track available agents
- **Task Dispatcher**: Route tasks to agents
- **Result Aggregator**: Combine results
- **Conflict Resolution**: Handle disagreements

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Callable
from enum import Enum
from datetime import datetime
import uuid

class AgentCapability(Enum):
    """Agent capabilities"""
    ANALYSIS = "analysis"
    DATA_PROCESSING = "data_processing"
    DECISION_MAKING = "decision_making"
    CODE_GENERATION = "code_generation"
    RESEARCH = "research"
    VALIDATION = "validation"

@dataclass
class TaskRequest:
    """Request for task execution"""
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    description: str = ""
    required_capability: AgentCapability = None
    priority: int = 5  # 1-10, higher is more urgent
    deadline: Optional[str] = None
    parameters: Dict[str, Any] = field(default_factory=dict)
    status: str = "pending"
    assigned_agent: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

@dataclass
class TaskResult:
    """Result from task execution"""
    task_id: str
    agent_id: str
    success: bool
    result: Any
    execution_time: float
    metadata: Dict[str, Any] = field(default_factory=dict)

class AgentProfile:
    """Profile of an agent in the system"""
    
    def __init__(self, agent_id: str, name: str, capabilities: List[AgentCapability]):
        self.agent_id = agent_id
        self.name = name
        self.capabilities = capabilities
        self.status = "available"
        self.current_task: Optional[str] = None
        self.completed_tasks = 0
        self.success_rate = 1.0
        self.average_execution_time = 0.0
        self.last_seen = datetime.now()
    
    def can_handle(self, capability: AgentCapability) -> bool:
        return capability in self.capabilities
    
    def is_available(self) -> bool:
        return self.status == "available"

class MultiAgentOrchestrator:
    """Orchestrates multiple agents"""
    
    def __init__(self):
        self.agents: Dict[str, AgentProfile] = {}
        self.task_queue: List[TaskRequest] = []
        self.task_results: List[TaskResult] = []
        self.task_assignments: Dict[str, str] = {}  # task_id -> agent_id
    
    def register_agent(self, agent: AgentProfile):
        """Register agent in system"""
        self.agents[agent.agent_id] = agent
        print(f"✓ Agent registered: {agent.name}")
    
    def submit_task(self, task: TaskRequest):
        """Submit task for execution"""
        self.task_queue.append(task)
        print(f"📋 Task submitted: {task.id} ({task.description})")
    
    def find_suitable_agent(self, task: TaskRequest) -> Optional[AgentProfile]:
        """Find best agent for task"""
        suitable_agents = [
            agent for agent in self.agents.values()
            if agent.can_handle(task.required_capability) and agent.is_available()
        ]
        
        if not suitable_agents:
            return None
        
        # Sort by success rate (highest first)
        suitable_agents.sort(key=lambda a: a.success_rate, reverse=True)
        return suitable_agents[0]
    
    def dispatch_task(self, task: TaskRequest) -> bool:
        """Dispatch task to suitable agent"""
        agent = self.find_suitable_agent(task)
        
        if not agent:
            print(f"❌ No suitable agent found for task {task.id}")
            task.status = "pending_no_agent"
            return False
        
        # Assign task
        task.assigned_agent = agent.agent_id
        task.status = "assigned"
        agent.current_task = task.id
        agent.status = "busy"
        
        self.task_assignments[task.id] = agent.agent_id
        print(f"✓ Task {task.id} assigned to {agent.name}")
        return True
    
    def execute_task(self, task_id: str) -> bool:
        """Execute assigned task"""
        if task_id not in self.task_assignments:
            return False
        
        agent_id = self.task_assignments[task_id]
        agent = self.agents[agent_id]
        
        # Simulate execution
        execution_time = 2.5
        success = True
        
        result = TaskResult(
            task_id=task_id,
            agent_id=agent_id,
            success=success,
            result="Task completed successfully",
            execution_time=execution_time
        )
        
        self.task_results.append(result)
        
        # Update agent stats
        agent.completed_tasks += 1
        agent.status = "available"
        agent.current_task = None
        
        print(f"✓ Task {task_id} completed by {agent.name} in {execution_time}s")
        return True
    
    def dispatch_and_execute_all(self):
        """Process all pending tasks"""
        print(f"\n🚀 Processing {len(self.task_queue)} tasks...\n")
        
        for task in self.task_queue:
            if self.dispatch_task(task):
                self.execute_task(task.id)
    
    def get_system_status(self) -> Dict[str, Any]:
        """Get overall system status"""
        return {
            "total_agents": len(self.agents),
            "available_agents": sum(1 for a in self.agents.values() if a.is_available()),
            "pending_tasks": len([t for t in self.task_queue if t.status == "pending"]),
            "completed_tasks": len(self.task_results),
            "total_execution_time": sum(r.execution_time for r in self.task_results)
        }

# Example usage
print("🔄 Multi-Agent Orchestration System")
print("="*50)

orchestrator = MultiAgentOrchestrator()

# Register agents
agents_config = [
    ("analyst_1", "Data Analyst", [AgentCapability.ANALYSIS, AgentCapability.DATA_PROCESSING]),
    ("dev_1", "Developer", [AgentCapability.CODE_GENERATION, AgentCapability.VALIDATION]),
    ("researcher_1", "Researcher", [AgentCapability.RESEARCH, AgentCapability.ANALYSIS]),
]

for agent_id, name, capabilities in agents_config:
    agent = AgentProfile(agent_id, name, capabilities)
    orchestrator.register_agent(agent)

print()

# Submit tasks
tasks_config = [
    ("Analyze customer data", AgentCapability.ANALYSIS),
    ("Generate API code", AgentCapability.CODE_GENERATION),
    ("Research market trends", AgentCapability.RESEARCH),
    ("Process data pipeline", AgentCapability.DATA_PROCESSING),
]

for description, capability in tasks_config:
    task = TaskRequest(
        description=description,
        required_capability=capability,
        priority=5
    )
    orchestrator.submit_task(task)

# Execute all
orchestrator.dispatch_and_execute_all()

# Print system status
status = orchestrator.get_system_status()
print(f"\n📊 System Status:")
for key, value in status.items():
    print(f"  {key}: {value}")

## 2. LLM-Powered Agent Architecture

Using Large Language Models as agent brains:
- **LLM as Decision Engine**: Model makes decisions
- **Prompt Engineering**: Structured prompts for actions
- **Function Calling**: LLM selects tools
- **Context Management**: Maintain conversation history

In [ ]:
@dataclass
class Message:
    """Message in agent conversation"""
    role: str  # "user", "assistant", "system"
    content: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    tool_calls: List[Dict] = field(default_factory=list)

class LLMAgentBrain:
    """LLM-based agent brain"""
    
    def __init__(self, model_name: str = "gpt-4", system_prompt: str = ""):
        self.model_name = model_name
        self.system_prompt = system_prompt or self._get_default_system_prompt()
        self.conversation_history: List[Message] = []
        self.available_functions = self._init_functions()
    
    def _get_default_system_prompt(self) -> str:
        return """You are a helpful AI agent. You have access to various tools and functions.
When you need to take action, clearly state which tool you're using and why.
Always think through problems step by step.
Provide clear reasoning for your decisions."""
    
    def _init_functions(self) -> Dict[str, Callable]:
        """Initialize available functions"""
        return {
            "search": self._func_search,
            "calculate": self._func_calculate,
            "summarize": self._func_summarize,
            "retrieve_data": self._func_retrieve_data,
        }
    
    def _func_search(self, query: str) -> str:
        return f"Search results for '{query}': Found 5 relevant results"
    
    def _func_calculate(self, expression: str) -> str:
        try:
            result = eval(expression)
            return f"Calculation result: {expression} = {result}"
        except:
            return f"Error calculating: {expression}"
    
    def _func_summarize(self, text: str) -> str:
        return f"Summary: {text[:50]}... [summarized]"
    
    def _func_retrieve_data(self, query: str) -> str:
        return f"Retrieved data for '{query}': [data_here]"
    
    def add_user_message(self, content: str):
        """Add user message to history"""
        msg = Message(role="user", content=content)
        self.conversation_history.append(msg)
    
    def add_assistant_message(self, content: str, tool_calls: List[Dict] = None):
        """Add assistant message to history"""
        msg = Message(
            role="assistant",
            content=content,
            tool_calls=tool_calls or []
        )
        self.conversation_history.append(msg)
    
    def process_request(self, user_input: str) -> str:
        """Process user request using LLM"""
        self.add_user_message(user_input)
        
        # Simulate LLM thinking
        thought_process = f"Thinking about: {user_input[:40]}..."
        
        # Determine which functions to call
        tool_calls = self._determine_tool_calls(user_input)
        
        # Execute tools
        results = []
        for tool_call in tool_calls:
            result = self._execute_tool(tool_call)
            results.append(result)
        
        # Generate response
        response = self._generate_response(user_input, results)
        
        self.add_assistant_message(response, tool_calls)
        
        return response
    
    def _determine_tool_calls(self, user_input: str) -> List[Dict]:
        """Determine which tools to call based on input"""
        tool_calls = []
        
        if "search" in user_input.lower():
            tool_calls.append({
                "name": "search",
                "arguments": {"query": user_input}
            })
        
        if any(op in user_input for op in ["calculate", "+", "-", "*", "/"]):
            tool_calls.append({
                "name": "calculate",
                "arguments": {"expression": "2+2"}
            })
        
        if "retrieve" in user_input.lower():
            tool_calls.append({
                "name": "retrieve_data",
                "arguments": {"query": user_input}
            })
        
        return tool_calls
    
    def _execute_tool(self, tool_call: Dict) -> str:
        """Execute tool call"""
        tool_name = tool_call["name"]
        args = tool_call["arguments"]
        
        if tool_name in self.available_functions:
            func = self.available_functions[tool_name]
            # Simplified argument passing
            if "query" in args:
                return func(args["query"])
            elif "expression" in args:
                return func(args["expression"])
        
        return f"Tool '{tool_name}' not available"
    
    def _generate_response(self, user_input: str, tool_results: List[str]) -> str:
        """Generate response from tool results"""
        if tool_results:
            return f"Based on the following information:\n" + "\n".join(tool_results) + "\nI can help you with that."
        else:
            return f"I understood your request about '{user_input[:30]}...'. How can I help you further?"
    
    def get_conversation_history(self) -> List[Dict]:
        """Get conversation history"""
        return [
            {"role": msg.role, "content": msg.content}
            for msg in self.conversation_history
        ]

# Example usage
print("🧠 LLM-Powered Agent")
print("="*50)

agent = LLMAgentBrain(
    model_name="gpt-4",
    system_prompt="You are an AI assistant with access to tools for data analysis."
)

# Interactive conversation
requests = [
    "Search for information about agentic AI",
    "Calculate 2 to the power of 10",
    "Retrieve customer data for analysis",
]

for user_request in requests:
    print(f"\n👤 User: {user_request}")
    response = agent.process_request(user_request)
    print(f"🤖 Agent: {response}")

print(f"\n📝 Conversation History ({len(agent.conversation_history)} messages):")
for i, msg in enumerate(agent.conversation_history, 1):
    print(f"  {i}. {msg.role}: {msg.content[:50]}...")

## 3. Agent Memory Systems

Persistent memory for agents:
- **Short-term Memory**: Current conversation
- **Long-term Memory**: Learned facts
- **Working Memory**: Current context
- **Memory Retrieval**: Semantic search

In [ ]:
@dataclass
class Memory:
    """Single memory item"""
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    content: str = ""
    memory_type: str = "fact"  # fact, experience, skill, relationship
    importance: float = 0.5  # 0-1 scale
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    accessed_count: int = 0
    embedding: List[float] = field(default_factory=list)  # For semantic search

class AgentMemorySystem:
    """Agent memory management"""
    
    def __init__(self, short_term_capacity: int = 20, long_term_capacity: int = 1000):
        self.short_term_memory: deque = deque(maxlen=short_term_capacity)
        self.long_term_memory: List[Memory] = []
        self.long_term_capacity = long_term_capacity
        self.working_memory: Dict[str, Any] = {}
    
    def add_short_term_memory(self, content: str, importance: float = 0.5):
        """Add to short-term memory"""
        mem = Memory(
            content=content,
            memory_type="experience",
            importance=importance
        )
        self.short_term_memory.append(mem)
        return mem.id
    
    def add_long_term_memory(self, content: str, memory_type: str = "fact", importance: float = 0.7):
        """Add to long-term memory"""
        # If at capacity, remove least important
        if len(self.long_term_memory) >= self.long_term_capacity:
            least_important = min(
                self.long_term_memory,
                key=lambda m: m.importance * m.accessed_count
            )
            self.long_term_memory.remove(least_important)
        
        mem = Memory(
            content=content,
            memory_type=memory_type,
            importance=importance
        )
        self.long_term_memory.append(mem)
        return mem.id
    
    def consolidate_memories(self):
        """Move important short-term to long-term"""
        for mem in list(self.short_term_memory):
            if mem.importance > 0.7:
                self.add_long_term_memory(mem.content, mem.memory_type, mem.importance)
    
    def retrieve_memory(self, query: str, top_k: int = 5) -> List[Memory]:
        """Retrieve memories matching query"""
        # Simple text matching (in production would use embeddings)
        query_words = set(query.lower().split())
        
        memories = []
        
        # Search short-term first
        for mem in self.short_term_memory:
            mem_words = set(mem.content.lower().split())
            similarity = len(query_words & mem_words) / len(query_words | mem_words)
            if similarity > 0.2:
                memories.append((mem, similarity))
        
        # Then long-term
        for mem in self.long_term_memory:
            mem_words = set(mem.content.lower().split())
            similarity = len(query_words & mem_words) / len(query_words | mem_words)
            if similarity > 0.2:
                memories.append((mem, similarity))
        
        # Sort by similarity and importance
        memories.sort(
            key=lambda x: (x[1] * x[0].importance),
            reverse=True
        )
        
        return [mem for mem, _ in memories[:top_k]]
    
    def set_working_memory(self, key: str, value: Any):
        """Set working memory item"""
        self.working_memory[key] = value
    
    def get_working_memory(self, key: str) -> Any:
        """Get working memory item"""
        return self.working_memory.get(key)
    
    def get_memory_status(self) -> Dict[str, Any]:
        """Get memory system status"""
        return {
            "short_term_items": len(self.short_term_memory),
            "long_term_items": len(self.long_term_memory),
            "working_memory_items": len(self.working_memory),
            "long_term_capacity": self.long_term_capacity,
            "long_term_usage": f"{len(self.long_term_memory)/self.long_term_capacity*100:.1f}%"
        }

# Example usage
print("💾 Agent Memory System")
print("="*50)

memory_sys = AgentMemorySystem()

# Add short-term memories
print("\n📝 Adding short-term memories:")
memory_sys.add_short_term_memory("User asked about agentic AI", importance=0.8)
memory_sys.add_short_term_memory("Discussed ReAct pattern", importance=0.7)
memory_sys.add_short_term_memory("Explained tool use", importance=0.9)
print("  ✓ 3 short-term memories added")

# Add long-term memories
print("\n🧠 Adding long-term memories:")
memory_sys.add_long_term_memory("Agentic AI uses autonomous decision-making", "fact", 0.9)
memory_sys.add_long_term_memory("Python is used for agent implementation", "fact", 0.8)
memory_sys.add_long_term_memory("Agents can use external tools", "fact", 0.9)
print("  ✓ 3 long-term memories added")

# Consolidate
print("\n🔄 Consolidating memories...")
memory_sys.consolidate_memories()
print("  ✓ Important short-term memories promoted")

# Retrieve
print("\n🔍 Memory retrieval test:")
query = "agentic AI"
results = memory_sys.retrieve_memory(query)
print(f"  Query: '{query}'")
print(f"  Found {len(results)} relevant memories:")
for i, mem in enumerate(results, 1):
    print(f"    {i}. {mem.content} (importance: {mem.importance})")

# Working memory
print("\n📌 Working memory:")
memory_sys.set_working_memory("current_task", "Learning agentic AI")
memory_sys.set_working_memory("user_preference", "detailed explanations")
print(f"  Current task: {memory_sys.get_working_memory('current_task')}")
print(f"  User preference: {memory_sys.get_working_memory('user_preference')}")

# Status
print(f"\n📊 Memory Status:")
status = memory_sys.get_memory_status()
for key, value in status.items():
    print(f"  {key}: {value}")

## 4. Agent Communication Protocol

Structured communication between agents:
- **Message Format**: Standardized structure
- **Protocol Rules**: Communication rules
- **Message Routing**: Directed communication
- **Acknowledgment**: Confirm receipt

In [ ]:
class MessageType(Enum):
    REQUEST = "request"
    RESPONSE = "response"
    NOTIFICATION = "notification"
    QUERY = "query"
    RESULT = "result"
    ERROR = "error"

@dataclass
class AgentMessage:
    """Structured message between agents"""
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    from_agent: str = ""
    to_agent: str = ""
    message_type: MessageType = MessageType.REQUEST
    content: Dict[str, Any] = field(default_factory=dict)
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    reply_to: Optional[str] = None
    status: str = "pending"  # pending, sent, received, processed

class AgentCommunicationHub:
    """Hub for agent-to-agent communication"""
    
    def __init__(self):
        self.message_log: List[AgentMessage] = []
        self.agent_inboxes: Dict[str, deque] = {}  # agent_id -> message queue
        self.delivery_receipts: Dict[str, str] = {}  # message_id -> delivery status
    
    def register_agent(self, agent_id: str):
        """Register agent for communication"""
        if agent_id not in self.agent_inboxes:
            self.agent_inboxes[agent_id] = deque()
    
    def send_message(self, message: AgentMessage) -> bool:
        """Send message from one agent to another"""
        if message.to_agent not in self.agent_inboxes:
            return False
        
        message.status = "sent"
        self.message_log.append(message)
        self.agent_inboxes[message.to_agent].append(message)
        
        print(f"📤 Message sent: {message.from_agent} → {message.to_agent}")
        return True
    
    def receive_message(self, agent_id: str) -> Optional[AgentMessage]:
        """Receive next message for agent"""
        if agent_id in self.agent_inboxes and self.agent_inboxes[agent_id]:
            message = self.agent_inboxes[agent_id].popleft()
            message.status = "received"
            print(f"📥 Message received: {agent_id}")
            return message
        return None
    
    def create_response(self, request: AgentMessage, response_content: Dict) -> AgentMessage:
        """Create response to request"""
        return AgentMessage(
            from_agent=request.to_agent,
            to_agent=request.from_agent,
            message_type=MessageType.RESPONSE,
            content=response_content,
            reply_to=request.id
        )
    
    def get_message_thread(self, initial_message_id: str) -> List[AgentMessage]:
        """Get conversation thread"""
        thread = []
        current_id = initial_message_id
        
        while current_id:
            msg = next((m for m in self.message_log if m.id == current_id), None)
            if msg:
                thread.append(msg)
                current_id = msg.reply_to
            else:
                break
        
        return thread
    
    def broadcast_message(self, from_agent: str, recipient_agents: List[str], 
                         message_type: MessageType, content: Dict):
        """Send message to multiple agents"""
        for recipient in recipient_agents:
            msg = AgentMessage(
                from_agent=from_agent,
                to_agent=recipient,
                message_type=message_type,
                content=content
            )
            self.send_message(msg)

# Example usage
print("📡 Agent Communication Protocol")
print("="*50)

hub = AgentCommunicationHub()

# Register agents
agents = ["agent_a", "agent_b", "agent_c"]
for agent_id in agents:
    hub.register_agent(agent_id)
print(f"✓ {len(agents)} agents registered\n")

# Send request
request_msg = AgentMessage(
    from_agent="agent_a",
    to_agent="agent_b",
    message_type=MessageType.REQUEST,
    content={"query": "Analyze this dataset", "priority": "high"}
)
hub.send_message(request_msg)

# Agent B receives and responds
received = hub.receive_message("agent_b")
if received:
    response = hub.create_response(received, {"result": "Analysis complete", "success": True})
    hub.send_message(response)

print()

# Broadcast notification
print("Broadcasting notification:")
hub.broadcast_message(
    from_agent="agent_a",
    recipient_agents=["agent_b", "agent_c"],
    message_type=MessageType.NOTIFICATION,
    content={"event": "system_update", "message": "New data available"}
)

print(f"\n📊 Communication Hub Status:")
print(f"  Total messages: {len(hub.message_log)}")
print(f"  Message types: {set(m.message_type.value for m in hub.message_log)}")

## 5. Error Handling and Resilience

Robust agent error handling:
- **Error Detection**: Identify failures
- **Recovery Strategies**: Retry, fallback
- **Graceful Degradation**: Reduced functionality
- **Logging and Monitoring**: Track issues

In [ ]:
class ErrorType(Enum):
    TOOL_ERROR = "tool_error"
    TIMEOUT = "timeout"
    INVALID_INPUT = "invalid_input"
    RESOURCE_EXHAUSTED = "resource_exhausted"
    EXTERNAL_SERVICE_ERROR = "external_service_error"
    UNKNOWN = "unknown"

@dataclass
class AgentError:
    """Error in agent execution"""
    error_type: ErrorType
    message: str
    context: Dict[str, Any] = field(default_factory=dict)
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    recoverable: bool = True
    suggested_action: Optional[str] = None

class ResilientAgent:
    """Agent with error handling and recovery"""
    
    def __init__(self, agent_id: str, max_retries: int = 3):
        self.agent_id = agent_id
        self.max_retries = max_retries
        self.error_log: List[AgentError] = []
        self.retry_count: Dict[str, int] = {}
    
    def execute_with_recovery(self, task_id: str, task_func: Callable, *args, **kwargs) -> Any:
        """Execute task with retry and recovery"""
        retry_count = 0
        last_error = None
        
        while retry_count < self.max_retries:
            try:
                print(f"⚙️  Executing task {task_id}...")
                result = task_func(*args, **kwargs)
                print(f"✓ Task {task_id} completed successfully")
                self.retry_count[task_id] = 0
                return result
            
            except Exception as e:
                retry_count += 1
                error = AgentError(
                    error_type=self._classify_error(str(e)),
                    message=str(e),
                    context={"task_id": task_id, "attempt": retry_count},
                    recoverable=retry_count < self.max_retries
                )
                last_error = error
                self.error_log.append(error)
                
                print(f"❌ Error on attempt {retry_count}: {str(e)}")
                
                if retry_count < self.max_retries:
                    print(f"🔄 Retrying... (attempt {retry_count + 1}/{self.max_retries})")
        
        # All retries failed
        if last_error:
            print(f"⚠️  Task {task_id} failed after {self.max_retries} attempts")
            return self._fallback_response(task_id, last_error)
    
    def _classify_error(self, error_message: str) -> ErrorType:
        """Classify error type"""
        error_msg_lower = error_message.lower()
        
        if "timeout" in error_msg_lower:
            return ErrorType.TIMEOUT
        elif "invalid" in error_msg_lower or "type" in error_msg_lower:
            return ErrorType.INVALID_INPUT
        elif "resource" in error_msg_lower or "memory" in error_msg_lower:
            return ErrorType.RESOURCE_EXHAUSTED
        elif "service" in error_msg_lower or "connection" in error_msg_lower:
            return ErrorType.EXTERNAL_SERVICE_ERROR
        else:
            return ErrorType.TOOL_ERROR
    
    def _fallback_response(self, task_id: str, error: AgentError) -> Dict[str, Any]:
        """Provide fallback response when task fails"""
        fallbacks = {
            ErrorType.TIMEOUT: "Operation timed out. Please try with smaller input.",
            ErrorType.RESOURCE_EXHAUSTED: "System resources low. Reducing scope.",
            ErrorType.EXTERNAL_SERVICE_ERROR: "External service unavailable. Using cached data.",
            ErrorType.INVALID_INPUT: "Invalid input. Please provide correct format.",
        }
        
        return {
            "status": "partial_failure",
            "error_type": error.error_type.value,
            "fallback_message": fallbacks.get(error.error_type, "Operation could not complete."),
            "task_id": task_id
        }
    
    def get_error_summary(self) -> Dict[str, Any]:
        """Get error summary"""
        error_types_count = {}
        for error in self.error_log:
            error_type = error.error_type.value
            error_types_count[error_type] = error_types_count.get(error_type, 0) + 1
        
        return {
            "total_errors": len(self.error_log),
            "error_types": error_types_count,
            "recoverable_errors": sum(1 for e in self.error_log if e.recoverable),
            "unrecoverable_errors": sum(1 for e in self.error_log if not e.recoverable)
        }

# Example usage
print("🛡️ Resilient Agent with Error Handling")
print("="*50)

agent = ResilientAgent(agent_id="resilient_1", max_retries=3)

def failing_task():
    """Simulated failing task"""
    import random
    if random.random() > 0.6:
        return {"status": "success", "result": "Task completed"}
    else:
        raise Exception("Simulated service timeout")

# Execute task with recovery
print("\nExecuting task with retry logic:")
result = agent.execute_with_recovery("task_001", failing_task)
print(f"Result: {result}")

print(f"\n📊 Error Summary:")
summary = agent.get_error_summary()
for key, value in summary.items():
    print(f"  {key}: {value}")

## Summary: Advanced Agent Structures

| Structure | Purpose | Key Components | When to Use |
|-----------|---------|-----------------|-------------|
| Multi-Agent Orchestration | Manage multiple agents | Task dispatcher, agent registry, result aggregator | Complex problems requiring specialization |
| LLM-Powered Brain | Use LLM as decision engine | Prompt, function calling, tool integration | Natural language interaction |
| Memory System | Persistent learning | Short/long-term memory, retrieval, consolidation | Learning and adaptation |
| Communication Protocol | Agent coordination | Message format, routing, acknowledgment | Multi-agent systems |
| Error Handling | Resilience | Error classification, retry, fallback | Production systems |

## Production Considerations

✅ **Scalability**: Handle many agents and tasks  
✅ **Reliability**: Retry strategies and fallbacks  
✅ **Observability**: Logging and monitoring  
✅ **Performance**: Optimize tool execution  
✅ **Security**: Validate agent inputs/outputs  
✅ **Compliance**: Track and audit decisions